# Femicide Graph Pipeline v6

v6: dyadic edges only + **factor nodes** (star topology still present)

Key differences from v7:
- Factor nodes still exist as separate nodes with HAS_FACTOR edges
- No rf_* attributes on person nodes
- No P9 principle
- Three-state logic on HAS_FACTOR edges
- GRAPH_VARIANT parameter for descriptive/predictive

In [ ]:
import re, json, hashlib, datetime as dt
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import networkx as nx

DATA_PATH     = Path("Stat_FW.xlsx")
SHEET_IDX     = 0
CASE_ID       = "4200-73111-00001-22"
GRAPH_VARIANT = "descriptive"  # "descriptive" or "predictive"
assert GRAPH_VARIANT in {"descriptive", "predictive"}

EXPORT_DIR = Path.cwd() / f"exports_v6_{GRAPH_VARIANT}"
EXPORT_DIR.mkdir(exist_ok=True)

PLACEHOLDERS = {"not applicable", "none", "unknown", "not known", "n/a", "", "nan"}

df = pd.read_excel(DATA_PATH, sheet_name=SHEET_IDX)
subset = df[df["case_id"] == CASE_ID]
row = subset.iloc[0]
cid = str(row["case_id"])
case_id     = f"CASE_{cid}"
offender_id = f"PERSON_{cid}_O"
victim_id   = f"PERSON_{cid}_V"
print(f"Config: variant={GRAPH_VARIANT}, case={cid}")

## Helper functions

In [ ]:
nodes = {}
edges = {}

def norm_str(x):
    if x is None: return None
    if isinstance(x, float) and np.isnan(x): return None
    s = str(x).strip()
    return None if s.lower() in PLACEHOLDERS else s

def parse_bool(x):
    s = norm_str(x)
    if s is None: return None
    sl = s.lower()
    if sl in {"yes","y","true","1"} or sl.startswith("yes"): return True
    if sl in {"no","n","false","0"} or sl.startswith("no"):  return False
    return None

def slugify(s):
    v = norm_str(s)
    if v is None: return None
    return re.sub(r"[^a-z0-9]+", "_", v.lower()).strip("_")

def stable_edge_id(s, t, etype, ekey):
    return "E_" + hashlib.sha1(f"{s}||{t}||{etype}||{ekey}".encode()).hexdigest()[:16]

def add_attr(d, k, v):
    v2 = norm_str(v)
    if v2 is not None: d[k] = v2

def add_node(nid, ntype, **attrs):
    n = nodes.get(nid, {"node_id": nid, "node_type": ntype})
    for k, v in attrs.items(): add_attr(n, k, v)
    nodes[nid] = n; return nid

def merge_edge(src, tgt, etype, ekey=None, **attrs):
    if ekey is None: ekey = etype
    eid = stable_edge_id(src, tgt, etype, ekey)
    e = edges.get(eid, {"edge_id": eid, "source_id": src, "target_id": tgt, "edge_type": etype, "edge_key": ekey})
    for k, v in attrs.items(): add_attr(e, k, v)
    edges[eid] = e; return eid

def merge_edge_threestate(src, tgt, etype, bval, ekey=None, **attrs):
    if bval is True:  attrs["value"] = 1; attrs["observed"] = 1
    elif bval is False: attrs["value"] = 0; attrs["observed"] = 1
    else: attrs["observed"] = 0
    return merge_edge(src, tgt, etype, ekey=ekey, **attrs)

print("Helpers loaded.")

## Core nodes + dyadic state

In [ ]:
add_node(case_id, "case", label=cid)
for k in ["crime_date","verdict_date","Crime_verdict_timegap","crime_arrest_timegap",
          "case_solved","case_type","court_number","internal_police_number"]:
    add_attr(nodes[case_id], k, row.get(k))

add_node(offender_id, "person", label="Offender", role="offender",
         age=row.get("offender_age"), gender=row.get("offender_gender"))
add_node(victim_id, "person", label="Victim", role="victim",
         age=row.get("victim_age"), gender=row.get("victim_gender"))

merge_edge(case_id, offender_id, "HAS_PARTICIPANT", role="offender", scope="case")
merge_edge(case_id, victim_id,   "HAS_PARTICIPANT", role="victim",   scope="case")

merge_edge_threestate(victim_id, offender_id, "SEPARATED_FROM",
                      parse_bool(row.get("separated")), scope="case", stage="prior")
merge_edge_threestate(victim_id, offender_id, "HAS_NEW_PARTNER",
                      parse_bool(row.get("victim_new_partner")), scope="case", stage="prior")
custody = parse_bool(row.get("child_custody_access_disputes"))
merge_edge_threestate(offender_id, victim_id, "CHILD_CUSTODY_DISPUTE", custody, scope="case", stage="prior")
merge_edge_threestate(victim_id, offender_id, "CHILD_CUSTODY_DISPUTE", custody, scope="case", stage="prior")
shared = parse_bool(row.get("shared_children"))
merge_edge_threestate(offender_id, victim_id, "SHARED_CHILDREN", shared, scope="case", stage="prior")
merge_edge_threestate(victim_id, offender_id, "SHARED_CHILDREN", shared, scope="case", stage="prior")

print(f"Core nodes: {len(nodes)}, edges: {len(edges)}")

## Factor nodes (P9 NOT yet applied — star topology)

In v6, risk factors are separate `factor` nodes connected via HAS_FACTOR edges.

In [ ]:
FACTOR_CANON = {
    "prior_suicide_attempt":    "suicide_attempt_prior",
    "offender_suicide_attempt": "suicide_attempt_during",
    "prior_suicide_threats":    "suicide_threats_prior",
    "threats_of_suicide":       "suicide_threats_during",
    "offender_suicide":         "suicide_post_incident",
}
FACTOR_STAGE_DEFAULT = {"offender_suicide": "post", "offender_suicide_attempt": "during"}

DYADIC_BEHAVIOR_MAP = {
    "escalation_of_violence":              "ESCALATED_VIOLENCE",
    "obsessive_behaviour":                 "STALKED",
    "sexual_jealousy":                     "SEXUAL_JEALOUSY",
    "misogynistic_attitudes":              "MISOGYNISTIC_ATTITUDES",
    "controlled_victims_daily_activities": "CONTROLLED_DAILY_ACTIVITIES",
    "threats_of_suicide":                  "THREATENED_SUICIDE",
}
DYADIC_SYMMETRIC = {"youth_couple": "YOUTH_COUPLE"}
RISK_HOLDER_MAP = {
    "offender_history_violence_outside_family": "offender",
    "prior_suicide_attempt": "offender", "excessive_alcohol_drug_use": "offender",
    "access_or_possession_firearms": "offender", "offender_suicide": "offender",
    "offender_suicide_attempt": "offender",
    "victim_considered_vulnerable": "victim", "victim_pregnant": "victim",
    "victim_disability": "victim",
}

FACTOR_AS_ATTR_COLS = [
    "offender_history_violence_outside_family","offender_history_domestic_violence_current",
    "offender_history_domestic_violence_past","prior_threats_to_kill_other",
    "prior_suicide_attempt","prior_suicide_threats","prior_sexual_assault_others",
    "excessive_alcohol_drug_use","offender_depressed_family_opinion","offender_depressed_professional",
    "access_or_possession_firearms","offender_suicide","offender_suicide_attempt",
    "victim_considered_vulnerable","victim_pregnant","victim_disability",
    "offender_access_to_victim_after_assessment","prior_hostage_taking",
    "prior_destruction_of_property","prior_violence_against_pets","prior_assault_while_pregnant",
]

def canonical_factor_key(col):
    key = FACTOR_CANON.get(col, col).lower()
    key = re.sub(r"^(offender|victim)_", "", key)
    key = re.sub(r"^prior_", "", key)
    return key

def infer_stage(raw, col):
    if col in FACTOR_STAGE_DEFAULT: return FACTOR_STAGE_DEFAULT[col]
    if col.startswith("prior_") or "history" in col: return "prior"
    return "prior"

PREDICTIVE_EXCLUDED = {"during", "post"}

for col in FACTOR_AS_ATTR_COLS:
    raw = row.get(col)
    b = parse_bool(raw)
    fkey = canonical_factor_key(col)
    stage = infer_stage(raw, col)

    if GRAPH_VARIANT == "predictive" and stage in PREDICTIVE_EXCLUDED:
        continue

    if col in DYADIC_BEHAVIOR_MAP:
        merge_edge_threestate(offender_id, victim_id, DYADIC_BEHAVIOR_MAP[col], b,
                              scope="case", stage=stage, source_col=col)
        continue

    # v6: create factor node + HAS_FACTOR edge
    sl = slugify(fkey)
    if sl:
        factor_id = f"FACTOR_{sl.upper()}"
        add_node(factor_id, "factor", label=fkey.replace("_"," ").title(),
                 factor_key=fkey, factor_type="risk_factor")
        holder = RISK_HOLDER_MAP.get(col, "offender")
        holder_id = offender_id if holder == "offender" else victim_id
        merge_edge_threestate(holder_id, factor_id, "HAS_FACTOR", b,
                              scope="case", stage=stage, source_col=col)

for col, etype in DYADIC_SYMMETRIC.items():
    b = parse_bool(row.get(col))
    merge_edge_threestate(offender_id, victim_id, etype, b, scope="case", stage="prior", source_col=col)
    merge_edge_threestate(victim_id, offender_id, etype, b, scope="case", stage="prior", source_col=col)

print(f"Factors done. Nodes: {len(nodes)} (including factor nodes), edges: {len(edges)}")

## Export

In [ ]:
nodes_df = pd.DataFrame(list(nodes.values()))
edges_df = pd.DataFrame(list(edges.values()))
print(f"Nodes: {len(nodes_df)}, Edges: {len(edges_df)}")
print("Node types:", nodes_df["node_type"].value_counts().to_dict())

G = nx.MultiDiGraph()
for _, n in nodes_df.iterrows():
    G.add_node(n["node_id"], **{k: v for k, v in n.items() if pd.notna(v)})
for _, e in edges_df.iterrows():
    attrs = {k: v for k, v in e.items() if k not in {"source_id","target_id"} and pd.notna(v)}
    G.add_edge(e["source_id"], e["target_id"], weight=1.0, **attrs)

nodes_df.to_csv(EXPORT_DIR / f"case_{cid}_nodes.csv", index=False)
edges_df.to_csv(EXPORT_DIR / f"case_{cid}_edges.csv", index=False)
nx.write_gexf(G, EXPORT_DIR / f"case_{cid}.gexf")
print(f"Exported to {EXPORT_DIR}/")

print("\nNOTE: v6 still has factor nodes (star topology).")
print("v7 eliminates these by moving factors to rf_* node attributes (P9).")
print("v7 also adds PRIOR_FAMILY_COURT_INVOLVEMENT and the full set of victim rf_ keys.")